In [40]:
import pandas as pd

data = pd.read_csv('../data/train.csv')
print(data.shape)
data.head(10)

(227845, 32)


,id,Time,V1,V2,V3,V4,V5,V6,V7,V8,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,1,143352.0,1.955041,-0.380783,-0.315013,0.330155,-0.509374,-0.086197,-0.627978,0.035994,...,0.238197,0.968305,0.053208,-0.278602,-0.044999,-0.216780,0.045168,-0.047145,9.99,0
1,2,117173.0,-0.400975,-0.626943,1.555339,-2.017772,-0.107769,0.168310,0.017959,-0.401619,...,-0.153485,0.421703,0.113442,-1.004095,-1.176695,0.361924,-0.370469,-0.144792,45.90,0
2,3,149565.0,0.072509,0.820566,-0.561351,-0.709897,1.080399,-0.359429,0.787858,0.117276,...,-0.314638,-0.872959,0.083391,0.148178,-0.431459,0.119690,0.206395,0.070288,11.99,0
3,4,93670.0,-0.535045,1.014587,1.750679,2.769390,0.500089,1.002270,0.847902,-0.081323,...,0.063525,0.443431,-0.072754,0.448192,-0.655203,-0.181038,-0.093013,-0.064931,117.44,0
4,5,82655.0,-4.026938,1.897371,-0.429786,-0.029571,-0.855751,-0.480406,-0.435632,1.313760,...,-0.480691,-0.230369,0.250717,0.066399,0.470787,0.245335,0.286904,-0.322672,25.76,0
5,6,38121.0,-0.795091,1.061833,0.640795,0.594567,0.551601,0.097600,0.633501,0.083765,...,0.026523,0.321166,-0.246997,-0.779914,-0.160225,-0.236537,0.442701,0.268076,40.22,0
6,7,75491.0,1.174701,0.275037,0.364384,1.078177,-0.137806,-0.423289,0.089835,-0.090439,...,0.109522,0.400376,-0.095922,0.102617,0.616368,-0.269566,0.036606,0.021232,14.90,0
7,8,128362.0,0.119471,0.026912,0.025870,-0.638642,1.569986,1.551793,0.573425,0.192499,...,0.493721,1.870004,-0.079431,-1.905932,-0.616224,0.629626,-0.049543,-0.192198,44.30,0
8,9,93605.0,-0.685023,1.448337,0.167864,4.622496,1.383526,0.187905,1.847737,-1.222425,...,-0.279236,0.609252,0.265805,-0.142988,-1.094047,0.119107,-0.527683,-0.036831,60.09,0
9,10,143475.0,0.025493,-0.317142,0.883483,-2.527437,-0.532859,-0.858682,0.010320,-0.267504,...,-0.048195,0.379341,-0.098510,0.001591,-0.432567,-0.227720,0.188823,0.176748,10.00,0


In [49]:
average = data.loc[data['Class'] == 0, 'Amount'].mean()
fraud = data[data['Class'] == 1]

answer1 = (fraud['Amount'] > average).sum()
print(answer1)

127


In [43]:
# computing Mahalanobis distance
import numpy as np

cols = [f"V{i}" for i in range(1, 29)] + ['Amount']
x = fraud[cols].to_numpy()

mean_vector = x.mean(axis=0)
print(mean_vector)

cov_matrix = np.cov(x, rowvar=False)
cov_matrix_inv = np.linalg.inv(cov_matrix)
print(cov_matrix.shape)

diff = x - mean_vector
D2 = np.sum((diff @ cov_matrix_inv) * diff, axis = 1)
D = np.sqrt(D2)
print(D.shape)

answer2 = np.round(D.mean(), 2)

[-4.48394790e+00  3.54540490e+00 -6.87275059e+00  4.56079613e+00
 -2.99706085e+00 -1.43620305e+00 -5.42867975e+00  7.05381917e-01
 -2.56190389e+00 -5.62765819e+00  3.82703748e+00 -6.34085481e+00
 -4.85917897e-02 -7.05536059e+00 -9.56659547e-02 -4.16096891e+00
 -6.68598265e+00 -2.24434923e+00  6.95518359e-01  3.32663777e-01
  7.37850068e-01  1.31408114e-02 -9.00589219e-02 -1.06439641e-01
  2.55331610e-02  6.28209350e-02  1.89715525e-01  8.53622559e-02
  1.11800939e+02]
(29, 29)
(394,)


In [ ]:
# function to test different hyperparameters of XGBoost, compared by F1 score on class 1
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)

def cv_score(params, X, y):
	oof = np.zeros(len(y))
	for train_idx, val_idx in skf.split(X, y):
		model = XGBClassifier(**params, tree_method='hist', random_state=67, max_delta_step=1)
		model.fit(X[train_idx], y[train_idx])
		oof[val_idx] = model.predict_proba(X[val_idx])[:, 1]

	thresholds = np.linspace(0.05, 0.95, 91)
	f1s = [f1_score(y, oof >= t) for t in thresholds]
	best = int(np.argmax(f1s))
	return f1s[best], thresholds[best]

In [45]:
feature_cols = [c for c in data.columns if c not in ('id', 'Class')]
X = data[feature_cols].to_numpy()
y = data["Class"].to_numpy()

base_f1, base_th = cv_score({}, X, y)
print(base_f1, base_th)

0.6779661016949152 0.42999999999999994


In [46]:
import itertools

grid = {
	'max_depth': [5, 6, 7],
	'min_child_weight': [4, 5, 6]
}

results = []
for values in itertools.product(*grid.values()):
	params = dict(zip(grid.keys(), values))
	f1, thr = cv_score(params, X, y)
	results.append({**params, 'f1': f1, 'threshold': thr})
	print(params, round(f1, 4), thr)

results = pd.DataFrame(results).sort_values('f1', ascending=False)
results.head()

{'max_depth': 5, 'min_child_weight': 4} 0.0029 0.95
{'max_depth': 5, 'min_child_weight': 5} 0.8529 0.5099999999999999
{'max_depth': 5, 'min_child_weight': 6} 0.8405 0.3499999999999999
{'max_depth': 6, 'min_child_weight': 4} 0.0026 0.95
{'max_depth': 6, 'min_child_weight': 5} 0.8513 0.3999999999999999
{'max_depth': 6, 'min_child_weight': 6} 0.8458 0.4799999999999999
{'max_depth': 7, 'min_child_weight': 4} 0.0025 0.95
{'max_depth': 7, 'min_child_weight': 5} 0.8505 0.4099999999999999
{'max_depth': 7, 'min_child_weight': 6} 0.8458 0.37999999999999995


,max_depth,min_child_weight,f1,threshold
1,5,5,0.852861,0.51
4,6,5,0.851296,0.40
7,7,5,0.850543,0.41
8,7,6,0.845839,0.38
5,6,6,0.845839,0.48


In [54]:
best_model = XGBClassifier(max_depth=5, min_child_weight=5, tree_method='hist', random_state=67, max_delta_step=1)
best_model.fit(X, y)

test = pd.read_csv('../data/test.csv')
ids = test['id']
X_test = test[feature_cols].to_numpy()
pred = best_model.predict_proba(X_test)[:, 1]
labels = (pred >= 0.51).astype(int)

predictions = pd.DataFrame({
	'subtaskID': 3,
	'datapointID': test['id'],
	'answer': labels.astype(str)
})

predictions.head()

,subtaskID,datapointID,answer
0,3,227846,1
1,3,227847,0
2,3,227848,0
3,3,227849,0
4,3,227850,0


In [55]:
first_answers = pd.DataFrame({
	'subtaskID': [1, 2],
	'datapointID': [1, 1],
	'answer': [str(answer1), f"{answer2:.2f}"],
})

submission = pd.concat([first_answers, predictions], ignore_index=True)
submission.to_csv('submission.csv', index=False)